# Which Feature Network Makes a Good Perceptual Loss for LDCT?

Refactored Week 1 and Week 2 notebook (seed 42). It runs top to bottom and writes every
table that `paper_plan.md` asks for in Weeks 1 and 2, including the Compute Budget.

What changed against `perceptual_backbone_ldct.ipynb`:

| Area | Before | Now |
|---|---|---|
| Imports | `src/` and the repo root were both on `sys.path`, so `config` and `src.config` were two different modules; `run_config.py` imported a missing `src.unet` | only the repo root is on the path, everything is imported as `src.*`, the U-Net comes from `src.model` |
| Seeds | only `random.seed` and `torch.cuda.manual_seed`; the U-Net was built before seeding; loader shuffling was unseeded | `seed_everything` seeds Python, NumPy and PyTorch; every U-Net is built right after seeding, so all configurations and the calibration share one initialization; loaders shuffle from their own seeded generator |
| Calibration crop | fixed centre crop, shuffled loader, C0 given a weight | the same random 224 x 224 crop rule as training, fixed order, one crop sequence for every network, so $S_1$ is identical in every row; C0 stays at $\lambda = 0$ |
| Dose-network patches | uniform over the 512 x 512 slice | positions sampled inside the body mask |
| RadImageNet | key mapping tied to one checkpoint layout | loader accepts the common layouts and fails if any backbone tensor is not loaded |
| Training loop | total run time only | ms per step, epoch time, GPU hours, peak memory, non-finite loss count, per-epoch log |
| Records | summary tables only | per-slice screen CSVs, dose curves, parameter counts, environment, and one Markdown file with the plan tables |

Order note: Step 2.2 (train the CT dose network) runs before Step 2.1 (calibration),
because the calibration of C9 and C10 needs the trained network.

## 0. Run configuration
Edit this cell only. Nothing below hard-codes a path, a seed or a budget.

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SEED = 42

WORK_DIR = Path("/content") if IN_COLAB else Path.cwd()
# The folder that contains src/. A local checkout is used when the notebook sits in one.
REPO_DIR = Path.cwd() if (Path.cwd() / "src" / "config.py").exists() else WORK_DIR / "perceptual_backbone_ldct"
REPO_URL = "https://github.com/UnkindledTwo/perceptual_backbone_ldct"
DATA_ROOT = WORK_DIR / "minideeplesion"
DATASET_GDRIVE_ID = "1pIUT5beAkp17WCxyGPHZfHy4kEiJtD4y"
OUTPUT_DIR = WORK_DIR / "outputs" / "perceptual_backbone"
# Optional Google Drive folder that receives a zip of OUTPUT_DIR at the end (None = no backup).
DRIVE_DIR = Path("/content/drive/MyDrive/BIAI/ISBI2027_WEIGHTS") if IN_COLAB else None

# Protocol (paper_plan.md). Changing any of these means re-running every row.
EPOCHS = 30
BATCH_SIZE = 8
CROP_SIZE = 224
TARGET_INFLUENCE = 0.95            # Psi*, share of the perceptual term in the initial loss
BATCH_OVERRIDES = {}               # e.g. {"C7": (4, 2)} = batch 4 with 2-step accumulation

# CT dose network (Step 2.2)
DOSE_NET_EPOCHS = 10
DOSE_PATCH_SIZE = 128
DOSE_PATCHES_PER_SLICE = 4
BODY_THRESHOLD = 0.05              # gt_clipped value above which a pixel counts as tissue
MIN_BODY_FRACTION = 0.9            # share of a patch that must lie inside the body mask

# RadImageNet input normalisation as (mean, std). None keeps the ImageNet statistics used so far.
# Still to be confirmed against the weights source before the results freeze.
RADIMAGENET_STATS = None

# What to run
TRAIN_CONFIGS = ["C0", "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10"]
SCREEN_BOTH_MATCHINGS = True       # the plan screens every feature-map network with both matchings
FORCE_RERUN = False                # False: a finished run (run_summary.json present) is loaded, not retrained
DETERMINISTIC = True               # cudnn.deterministic; keep the same value for every run that is compared
NUM_WORKERS = 4

## 1. Setup: data, repository, packages, imports

In [ ]:
%pip install -q lpips piq huggingface_hub gdown

In [ ]:
# Dataset (5.7 GB) and repository are fetched only when they are missing.
if not DATA_ROOT.exists():
    !gdown {DATASET_GDRIVE_ID} -O {WORK_DIR}/dataset_biailab_I2I_2025v1.zip
    !unzip -q -o {WORK_DIR}/dataset_biailab_I2I_2025v1.zip -d {WORK_DIR}
if not (REPO_DIR / "src" / "config.py").exists():
    !git clone {REPO_URL} {REPO_DIR}
if IN_COLAB and DRIVE_DIR is not None:
    from google.colab import drive
    drive.mount("/content/drive")
assert DATA_ROOT.exists(), f"dataset not found at {DATA_ROOT}"
assert (REPO_DIR / "src" / "config.py").exists(), f"src/ not found under {REPO_DIR}"

In [ ]:
# One import cell. Only the repository root goes on sys.path, so every module has exactly one
# identity (src.config, src.metrics, ...). The old notebook also added src/ itself, which loaded
# `config` and `src.config` as two different modules and built the LPIPS and DISTS metrics twice.
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import json
import platform
import random
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from IPython.display import Markdown, display
from scipy import ndimage
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet50
from tqdm.auto import tqdm

from src.config import DOSE_KEYS, PerceptualConfig
from src.ct_dose_net import CTDoseNet, DoseNetFeatures
from src.data import I2IPairs, assert_no_patient_leakage, patient_id
from src.distances import FeatureDistance, PerceptualObjective, PiqDistance
from src.feature_nets import IMAGENET, FeatureNet, build_feature_net
from src.metrics import slice_metrics
from src.model import UNet
from src.screen import DOSES, screen_network
from src.train import validation_l1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_ROOT = OUTPUT_DIR / f"seed{SEED}"          # everything that depends on the seed
SCREEN_DIR = OUTPUT_DIR / "screen"             # per-slice screen results (seed free)
for folder in (OUTPUT_DIR, RUN_ROOT, SCREEN_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("device:", device, "| repo:", REPO_DIR, "| outputs:", OUTPUT_DIR)

In [ ]:
# Shared helpers: seeding, seeded loaders, timing, small persistent records.
def seed_everything(seed: int) -> None:
    """Seed Python, NumPy and PyTorch (CPU and every GPU) and fix the cuDNN mode."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = DETERMINISTIC
    torch.backends.cudnn.benchmark = not DETERMINISTIC


def _seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % 2 ** 32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(dataset: Dataset, batch_size: int, shuffle: bool = False, seed: int = SEED) -> DataLoader:
    """Loader whose shuffling and worker seeds come from its own generator, not the global RNG."""
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS,
                      generator=generator, worker_init_fn=_seed_worker)


def sync() -> None:
    if device.type == "cuda":
        torch.cuda.synchronize()


def count_params(module: nn.Module | None) -> int:
    return 0 if module is None else sum(p.numel() for p in module.parameters())


def free(*modules: nn.Module | None) -> None:
    for module in modules:
        if module is not None:
            module.cpu()
    if device.type == "cuda":
        torch.cuda.empty_cache()


TIMINGS_PATH = OUTPUT_DIR / "timings.json"


def record_seconds(name: str, seconds: float) -> None:
    """Wall-clock seconds of one stage, kept on disk so the Compute Budget survives a restart."""
    timings = json.loads(TIMINGS_PATH.read_text()) if TIMINGS_PATH.exists() else {}
    timings[name] = float(seconds)
    TIMINGS_PATH.write_text(json.dumps(timings, indent=2))


def to_markdown(frame: pd.DataFrame, digits: int = 4) -> str:
    def cell(value) -> str:
        if isinstance(value, (bool, np.bool_)):
            return "yes" if value else "no"
        if isinstance(value, (float, np.floating)):
            return "-" if not np.isfinite(value) else f"{value:.{digits}f}"
        return "-" if value is None else str(value)

    lines = ["| " + " | ".join(map(str, frame.columns)) + " |", "|" + "---|" * len(frame.columns)]
    lines += ["| " + " | ".join(cell(v) for v in row) + " |" for row in frame.itertuples(index=False)]
    return "\n".join(lines)

In [ ]:
# Environment record for the GitHub issue comment (commit hash, environment, data split).
def repo_commit() -> str:
    """Commit of the checkout, read from the .git folder (no git command needed)."""
    git_dir = REPO_DIR / ".git"
    try:
        head = (git_dir / "HEAD").read_text().strip()
        if not head.startswith("ref: "):
            return head
        ref = head.removeprefix("ref: ")
        if (git_dir / ref).exists():
            return (git_dir / ref).read_text().strip()
        for line in (git_dir / "packed-refs").read_text().splitlines():
            if line.endswith(" " + ref):
                return line.split()[0]
    except OSError:
        pass
    return "unknown"


environment = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 2 ** 30, 1) if device.type == "cuda" else 0.0,
    "cudnn_deterministic": DETERMINISTIC,
    "repo_commit": repo_commit(),
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "crop_size": CROP_SIZE,
    "target_influence": TARGET_INFLUENCE,
}
(OUTPUT_DIR / "environment.json").write_text(json.dumps(environment, indent=2))
environment

# Week 1: Data, Reference and the Training-Free Screen

## Step 1.1: Split check, loaders and metrics

In [ ]:
FILES = {split: sorted((DATA_ROOT / split).glob("*.npz")) for split in ("train", "val", "test")}
patients_per_split = assert_no_patient_leakage(DATA_ROOT)     # raises if a patient is in two splits
print("patients per split:", patients_per_split)
print("slices per split:  ", {split: len(files) for split, files in FILES.items()})

with np.load(FILES["train"][0]) as sample:
    print("arrays:", sample.files)

# Shape and value range of the training input and target over every slice of every split.
INPUT_KEY, TARGET_KEY = "noisy_clipped_100k", "gt_clipped"
shapes, low, high = set(), float("inf"), float("-inf")
for path in tqdm([p for files in FILES.values() for p in files], desc="range check"):
    with np.load(path) as sample:
        for key in (INPUT_KEY, TARGET_KEY):
            shapes.add(sample[key].shape)
            low, high = min(low, float(sample[key].min())), max(high, float(sample[key].max()))
print("shapes:", shapes, f"| value range: [{low:.3f}, {high:.3f}]")
assert shapes == {(512, 512)} and low >= 0.0 and high <= 1.0, "dataset does not match the Protocol"

In [ ]:
DATASETS = {split: I2IPairs(DATA_ROOT / split, INPUT_KEY, TARGET_KEY) for split in ("train", "val", "test")}

# Identity test: PSNR inf, SSIM 1, LPIPS 0, DISTS 0, R_H 1 are expected.
noisy_slice, clean_slice, first_stem = DATASETS["test"][0]
identity_metrics = slice_metrics(clean_slice[0].numpy(), clean_slice[0].numpy())
print("identity:", identity_metrics)
print("input   :", slice_metrics(noisy_slice[0].numpy(), clean_slice[0].numpy()))

### LDCT input row of Table 1 (noisy 100k vs normal dose, test split)

In [ ]:
baseline_rows = []
for noisy_slice, clean_slice, stem in tqdm(DATASETS["test"], desc="LDCT input"):
    baseline_rows.append({"slice_id": stem, "patient_id": patient_id(Path(stem)),
                          **slice_metrics(noisy_slice[0].numpy(), clean_slice[0].numpy())})
df_baseline = pd.DataFrame(baseline_rows)
df_baseline.to_csv(OUTPUT_DIR / "baseline_metrics.csv", index=False)

METRICS = ["psnr", "ssim", "lpips", "dists", "r_h"]
display(df_baseline[METRICS].agg(["mean", "std"]).T)

step_1_1 = pd.DataFrame([
    {"Check": "Train / val / test patients", "Expected": "disjoint", "Observed": " / ".join(str(patients_per_split[s]) for s in ("train", "val", "test"))},
    {"Check": "Train / val / test slices", "Expected": "899 / 182 / 220", "Observed": " / ".join(str(len(FILES[s])) for s in ("train", "val", "test"))},
    {"Check": "Shared patients between splits", "Expected": "0", "Observed": "0"},
    {"Check": "Input and target range", "Expected": "[0, 1]", "Observed": f"[{low:.3f}, {high:.3f}]"},
    {"Check": "Metrics on identity input", "Expected": "PSNR inf, SSIM 1, LPIPS 0, DISTS 0, R_H 1",
     "Observed": ", ".join(f"{k.upper()} {v:.3g}" for k, v in identity_metrics.items())},
    {"Check": "SSIM of noisy_clipped_100k vs gt_clipped, test mean", "Expected": "below the U-Net outputs",
     "Observed": f"{df_baseline['ssim'].mean():.4f}"},
])
step_1_1.to_csv(OUTPUT_DIR / "step1_1_checks.csv", index=False)
display(step_1_1)

### Crop sanity check
The screen uses the fixed centre crop (rows and columns 144 to 368). Training and calibration
use one random 224 x 224 crop per batch.

In [ ]:
noisy_slice, clean_slice, _ = DATASETS["val"][0]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
axes[0].imshow(noisy_slice[0], cmap="gray", vmin=0, vmax=1)
axes[0].add_patch(plt.Rectangle((144, 144), CROP_SIZE, CROP_SIZE, fill=False, color="tab:orange", lw=1.5))
axes[0].set_title("100k input, screen crop outlined")
axes[1].imshow(noisy_slice[0, 144:144 + CROP_SIZE, 144:144 + CROP_SIZE], cmap="gray", vmin=0, vmax=1)
axes[1].set_title("screen crop, input")
axes[2].imshow(clean_slice[0, 144:144 + CROP_SIZE, 144:144 + CROP_SIZE], cmap="gray", vmin=0, vmax=1)
axes[2].set_title("screen crop, normal dose")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

## Feature networks, objectives and the training loop
Defined once and used by the screen, the calibration and every training run, so the three
stages cannot drift apart.

In [ ]:
# id: (feature network key, matching, label for tables)
CONFIG_SPECS = {
    "C0": ("none", "pointwise", "L1"),
    "C1": ("vgg19", "pointwise", "VGG-19"),
    "C2": ("lpips_vgg", "pointwise", "LPIPS-VGG"),
    "C3": ("dists", "statistics", "DISTS"),
    "C4": ("radimagenet", "pointwise", "RadImageNet"),
    "C5": ("dinov2_s", "pointwise", "DINOv2-S pointwise"),
    "C6": ("dinov2_b", "pointwise", "DINOv2-B pointwise"),
    "C7": ("medsiglip", "pointwise", "MedSigLIP"),
    "C8": ("dinov2_s", "statistics", "DINOv2-S statistics"),
    "C9": ("ct_dose_net", "pointwise", "CT dose net pointwise"),
    "C10": ("ct_dose_net", "statistics", "CT dose net statistics"),
}
# Networks of the training-free screen: label, feature network key. DINOv2-L and CLIP are screen only.
ZOO = [
    ("VGG-19 relu3_2", "vgg19"),
    ("LPIPS-VGG", "lpips_vgg"),
    ("DISTS", "dists"),
    ("RadImageNet ResNet-50", "radimagenet"),
    ("DINOv2-S", "dinov2_s"),
    ("DINOv2-B", "dinov2_b"),
    ("DINOv2-L", "dinov2_l"),
    ("CLIP ViT-B/16", "clip_b16"),
    ("MedSigLIP-448", "medsiglip"),
]
FIXED_MATCHING = {"lpips_vgg": "pointwise", "dists": "statistics"}    # these metrics define their own matching
TRAINED_PAIRS = {(net, matching) for net, matching, _ in CONFIG_SPECS.values()}

DOSE_NET_DIR = RUN_ROOT / "ct_dose_net"
DOSE_NET_PATH = DOSE_NET_DIR / "ct_dose_net.pt"       # written in Step 2.2
RADIMAGENET_WEIGHTS = None                             # downloaded in Step 1.3

In [ ]:
# RadImageNet ResNet-50 loading.
# The Hugging Face checkpoint stores an nn.Sequential backbone ("backbone.0.weight", "backbone.4.0.conv1.weight"),
# torchvision expects "conv1.weight", "layer1.0.conv1.weight". The first notebook loaded it with strict=False,
# which silently left the network at its random initialization. Here every key is mapped, and the load
# fails loudly unless every backbone tensor of the ResNet-50 was filled from the checkpoint.
_SEQUENTIAL_TO_RESNET = {"0": "conv1", "1": "bn1", "4": "layer1", "5": "layer2", "6": "layer3", "7": "layer4"}
_WRAPPER_KEYS = ("state_dict", "model_state_dict", "model")
_PREFIXES = ("module.", "backbone.", "model.", "encoder.")


def _is_backbone_tensor(name: str) -> bool:
    return not name.startswith("fc.") and not name.endswith("num_batches_tracked")


def load_radimagenet_state(body: nn.Module, weights_path: Path) -> dict[str, int]:
    raw = torch.load(weights_path, map_location="cpu")
    if isinstance(raw, nn.Module):
        raw = raw.state_dict()
    for wrapper in _WRAPPER_KEYS:
        if isinstance(raw, dict) and isinstance(raw.get(wrapper), dict):
            raw = raw[wrapper]
    expected = body.state_dict()
    state, skipped = {}, []
    for key, value in raw.items():
        name = key
        while name.startswith(_PREFIXES):
            name = name.split(".", 1)[1]
        head, _, rest = name.partition(".")
        if head in _SEQUENTIAL_TO_RESNET and rest:
            name = f"{_SEQUENTIAL_TO_RESNET[head]}.{rest}"
        if name in expected and expected[name].shape == value.shape:
            state[name] = value
        else:
            skipped.append(key)
    result = body.load_state_dict(state, strict=False)
    missing = [name for name in result.missing_keys if _is_backbone_tensor(name)]
    if missing:
        raise RuntimeError(f"RadImageNet weights did not load: {len(missing)} backbone tensors missing, "
                           f"for example {missing[:5]}; checkpoint keys look like {list(raw)[:5]}")
    wanted = sum(_is_backbone_tensor(name) for name in expected)
    loaded = sum(_is_backbone_tensor(name) for name in state)
    print(f"RadImageNet: loaded {loaded} of {wanted} backbone tensors; "
          f"{len(skipped)} checkpoint keys not used {skipped[:4]}")
    return {"loaded": loaded, "expected": wanted, "skipped": len(skipped)}


class RadImageNetFeatures(FeatureNet):
    """ResNet-50 layer1 to layer3 features with verified RadImageNet weights."""

    def __init__(self, weights_path: Path, stats=None) -> None:
        super().__init__(224, stats or IMAGENET)
        self.body = resnet50(weights=None)
        self.load_report = load_radimagenet_state(self.body, weights_path)

    def features(self, image: torch.Tensor) -> list[torch.Tensor]:
        b = self.body
        x = b.maxpool(b.relu(b.bn1(b.conv1(self.prepare(image)))))
        f1 = b.layer1(x)
        f2 = b.layer2(f1)
        return [f1, f2, b.layer3(f2)]

In [ ]:
def make_feature_net(name: str) -> FeatureNet | None:
    """Feature network behind the common interface. None for the L1 reference and the piq metrics."""
    if name in ("none", "lpips_vgg", "dists"):
        return None
    if name == "ct_dose_net":
        network = CTDoseNet()
        network.load_state_dict(torch.load(DOSE_NET_PATH, map_location="cpu"))
        return DoseNetFeatures(network)
    if name == "radimagenet":
        assert RADIMAGENET_WEIGHTS is not None, "run the Step 1.3 download cell first"
        return RadImageNetFeatures(RADIMAGENET_WEIGHTS, RADIMAGENET_STATS)
    return build_feature_net(name)


def make_distance(name: str, matching: str, net: FeatureNet | None = None) -> nn.Module | None:
    """The distance d_phi used by the screen, the calibration and the training loss."""
    if name == "none":
        return None
    if name in ("lpips_vgg", "dists"):
        return PiqDistance(name)
    return FeatureDistance(net if net is not None else make_feature_net(name), matching)


def random_crop_pair(prediction: torch.Tensor, target: torch.Tensor, size: int,
                     generator: torch.Generator) -> tuple[torch.Tensor, torch.Tensor]:
    """Same rule as src.distances.shared_crop (one random crop per batch), drawn from an explicit generator."""
    top = torch.randint(0, prediction.shape[-2] - size + 1, (1,), generator=generator).item()
    left = torch.randint(0, prediction.shape[-1] - size + 1, (1,), generator=generator).item()
    return prediction[..., top:top + size, left:left + size], target[..., top:top + size, left:left + size]

In [ ]:
WARMUP_STEPS = 10      # first steps left out of the ms-per-step average (cuDNN setup, allocator)


def train_config(config_id: str, loss_weight: float = 0.0, seed: int = SEED) -> tuple[pd.DataFrame, dict]:
    """Train one configuration with the Protocol budget, test it, and record its compute cost.

    Same loop as src.train.train_and_test, plus seeding and timing. Returns the per-slice test
    metrics and the run summary. A finished run is loaded from disk unless FORCE_RERUN is set.
    """
    feature_net, matching, label = CONFIG_SPECS[config_id]
    batch_size, accumulation = BATCH_OVERRIDES.get(config_id, (BATCH_SIZE, 1))
    config = PerceptualConfig(run_name=f"train_{config_id}", data_root=DATA_ROOT, feature_net=feature_net,
                              matching=matching, loss_weight=float(loss_weight), output_dir=RUN_ROOT,
                              crop_size=CROP_SIZE, seed=seed, batch_size=batch_size,
                              grad_accumulation=accumulation, epochs=EPOCHS)
    run_dir = config.run_dir
    run_dir.mkdir(parents=True, exist_ok=True)
    summary_path, metrics_path, checkpoint = run_dir / "run_summary.json", run_dir / "metrics.csv", run_dir / "best.pth"
    if summary_path.exists() and metrics_path.exists() and not FORCE_RERUN:
        print(f"{config_id}: finished run found in {run_dir}, loading it")
        return pd.read_csv(metrics_path, dtype={"patient_id": str}), json.loads(summary_path.read_text())

    # 1. Seed, then build the U-Net before anything else draws random numbers: every configuration and
    #    the calibration start from the same initialization theta_0.
    seed_everything(seed)
    model = UNet().to(device)
    # 2. Building a feature network can consume random numbers (a ResNet is initialized before its weights
    #    are loaded), so the global RNG is reset afterwards. The random crops then follow the same
    #    sequence in every configuration; batch order comes from the loader's own generator.
    distance = make_distance(feature_net, matching)
    criterion = PerceptualObjective(distance, config.loss_weight, config.crop_size).to(device)
    seed_everything(seed)

    train_loader = make_loader(DATASETS["train"], config.batch_size, shuffle=True, seed=seed)
    val_loader = make_loader(DATASETS["val"], config.batch_size)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5)
    use_amp = config.use_amp and device.type == "cuda"
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    best, best_epoch, nonfinite, step_ms, log = float("inf"), -1, 0, [], []
    run_start = time.perf_counter()
    for epoch in tqdm(range(config.epochs), desc=config.run_name):
        model.train()
        epoch_start, running, seen = time.perf_counter(), 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        for step, (noisy, clean, _) in enumerate(train_loader):
            noisy, clean = noisy.to(device), clean.to(device)
            sync()
            step_start = time.perf_counter()
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                prediction = model(noisy)
            loss = criterion(prediction.float(), clean)       # perceptual term in float32
            if not torch.isfinite(loss):
                nonfinite += 1
                optimizer.zero_grad(set_to_none=True)
                continue
            scaler.scale(loss / config.grad_accumulation).backward()
            if (step + 1) % config.grad_accumulation == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            sync()
            step_ms.append((time.perf_counter() - step_start) * 1e3)
            running += loss.item() * noisy.size(0)
            seen += noisy.size(0)
        score = validation_l1(model, val_loader, device)
        scheduler.step(score)
        if score < best:                                      # lowest validation L1, for every configuration
            best, best_epoch = score, epoch + 1
            torch.save(model.state_dict(), checkpoint)
        log.append({"epoch": epoch + 1, "train_loss": running / max(seen, 1), "val_l1": score,
                    "lr": optimizer.param_groups[0]["lr"], "epoch_seconds": time.perf_counter() - epoch_start})
    train_seconds = time.perf_counter() - run_start
    pd.DataFrame(log).to_csv(run_dir / "train_log.csv", index=False)

    model.load_state_dict(torch.load(checkpoint, map_location=device))
    model.eval()
    rows, test_start = [], time.perf_counter()
    with torch.no_grad():
        for noisy, clean, stem in DATASETS["test"]:
            prediction = model(noisy[None].to(device)).clamp(0, 1)[0, 0].float().cpu().numpy()
            rows.append({"slice_id": stem, "patient_id": patient_id(Path(stem)),
                         **slice_metrics(prediction, clean[0].numpy())})
    test_seconds = time.perf_counter() - test_start
    frame = pd.DataFrame(rows)
    frame.to_csv(metrics_path, index=False)

    timed = step_ms[WARMUP_STEPS:] if len(step_ms) > WARMUP_STEPS else step_ms
    summary = {
        "config": config_id, "label": label, "feature_net": feature_net, "matching": matching,
        "loss_weight": config.loss_weight, "seed": seed, "epochs": config.epochs,
        "batch_size": config.batch_size, "grad_accumulation": config.grad_accumulation,
        "ms_per_step": float(np.mean(timed)) if timed else float("nan"),
        "ms_per_step_median": float(np.median(timed)) if timed else float("nan"),
        "steps": len(step_ms), "nonfinite_steps": nonfinite,
        "train_seconds": train_seconds, "test_seconds": test_seconds,
        "gpu_hours": (train_seconds + test_seconds) / 3600,
        "peak_gpu_memory_gb": torch.cuda.max_memory_allocated() / 2 ** 30 if device.type == "cuda" else 0.0,
        "best_val_l1": best, "best_epoch": best_epoch,
        "unet_parameters": count_params(model), "feature_parameters": count_params(distance),
        "n_test": len(frame),
        **{f"{m}_mean": float(frame[m].mean()) for m in METRICS},
        **{f"{m}_std": float(frame[m].std(ddof=1)) for m in METRICS},
    }
    summary_path.write_text(json.dumps(summary, indent=2))
    free(model, criterion)
    print(f"{config_id}: PSNR {summary['psnr_mean']:.2f} dB, R_H {summary['r_h_mean']:.3f}, "
          f"{summary['ms_per_step']:.0f} ms/step, {summary['gpu_hours']:.2f} GPU h, "
          f"{nonfinite} non-finite steps")
    return frame, summary

## Step 1.2: Train the L1 reference (C0)
Produces the C0 row of Table 1 and "GPU hours per U-Net run, C0" of the Compute Budget.

In [ ]:
RESULTS, SUMMARIES = {}, {}
if "C0" in TRAIN_CONFIGS:
    RESULTS["C0"], SUMMARIES["C0"] = train_config("C0", loss_weight=0.0)
    display(pd.DataFrame([{"Configuration": "LDCT input (100k)", **{m: df_baseline[m].mean() for m in METRICS}},
                          {"Configuration": "C0 L1", **{m: RESULTS["C0"][m].mean() for m in METRICS}}]))

## Step 1.3 and Step 1.4: Feature network zoo, the two distances and the training-free screen
Each network is built once. It is checked (distance 0 at equality, gradient reaches the
prediction, ms per batch of 8 at 224 x 224 forward and backward) and then screened on the
validation split (dose ordering $M_\phi$, dose curve $D_\phi(d)$, equal-MSE blur penalty $B_\phi$).
The per-slice screen results are saved; Fig 2 is drawn from them.

In [ ]:
from huggingface_hub import hf_hub_download, login

zoo_keys = {key for _, key in ZOO} | {CONFIG_SPECS[c][0] for c in TRAIN_CONFIGS}
if "medsiglip" in zoo_keys:
    login()        # google/medsiglip-448 is a gated model
if "radimagenet" in zoo_keys:
    RADIMAGENET_WEIGHTS = Path(hf_hub_download(repo_id="Lab-Rasool/RadImageNet", filename="ResNet50.pt"))
    print("RadImageNet weights:", RADIMAGENET_WEIGHTS)

In [ ]:
def check_distance(distance: nn.Module) -> dict:
    """Step 1.3 checks on a random 64 x 64 input."""
    x = torch.rand(1, 1, 64, 64, device=device)
    with torch.no_grad():
        at_equality = distance(x, x.clone()).item()
    prediction = torch.rand(1, 1, 64, 64, device=device, requires_grad=True)
    distance(prediction, torch.rand(1, 1, 64, 64, device=device)).backward()
    gradient = prediction.grad
    reaches = gradient is not None and bool(torch.isfinite(gradient).all()) and gradient.abs().sum().item() > 0
    return {"distance_at_equality": at_equality, "zero_at_equality": abs(at_equality) < 1e-5,
            "gradient_reaches_prediction": reaches}


def backward_ms(distance: nn.Module, batch: int = 8, size: int = 224, warmup: int = 2, repeats: int = 5) -> float:
    """ms per batch of 8 at 224 x 224, forward and backward through the feature distance."""
    times = []
    try:
        for index in range(warmup + repeats):
            prediction = torch.rand(batch, 1, size, size, device=device, requires_grad=True)
            target = torch.rand(batch, 1, size, size, device=device)
            sync()
            start = time.perf_counter()
            distance(prediction, target).backward()
            sync()
            if index >= warmup:
                times.append((time.perf_counter() - start) * 1e3)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return float("nan")
    return float(np.mean(times))


def screen_summary(frame: pd.DataFrame) -> dict:
    return {"ordering_M": frame["ordering"].mean(),
            "dose_ratio_10k_1000k": frame["d_10000"].mean() / frame["d_1000000"].mean(),
            "blur_penalty_B": frame["d_blur"].sum() / frame["d_100000"].sum(),
            **{f"D_{dose}": frame[f"d_{dose}"].mean() for dose in DOSES},
            "D_blur": frame["d_blur"].mean()}


ZOO_CHECKS, SCREEN_ROWS, FAILED = {}, {}, {}


def evaluate_network(label: str, name: str) -> None:
    """Steps 1.3 and 1.4 for one feature network, with every matching it supports."""
    if name in FIXED_MATCHING:
        matchings = [FIXED_MATCHING[name]]
    elif SCREEN_BOTH_MATCHINGS:
        matchings = ["pointwise", "statistics"]
    else:
        matchings = [m for m in ("pointwise", "statistics") if (name, m) in TRAINED_PAIRS] or ["pointwise"]
    net = make_feature_net(name)
    for matching in matchings:
        distance = make_distance(name, matching, net).to(device)
        seed_everything(SEED)
        ZOO_CHECKS[(label, matching)] = {
            "Network": label, "Matching": matching, **check_distance(distance),
            "ms_per_batch_of_8": backward_ms(distance), "Parameters": count_params(distance),
            "trained": (name, matching) in TRAINED_PAIRS}
        start = time.perf_counter()
        frame = screen_network(distance, FILES["val"], device, crop=CROP_SIZE)
        frame.to_csv(SCREEN_DIR / f"{name}_{matching}.csv", index=False)
        SCREEN_ROWS[(label, matching)] = {
            "Network": label, "Matching": matching, **screen_summary(frame),
            "Parameters": count_params(distance), "trained": (name, matching) in TRAINED_PAIRS,
            "screen_seconds": time.perf_counter() - start}
        free(distance)
    free(net)


def save_zoo_reports() -> tuple[pd.DataFrame, pd.DataFrame]:
    checks, screen = pd.DataFrame(ZOO_CHECKS.values()), pd.DataFrame(SCREEN_ROWS.values())
    checks.to_csv(OUTPUT_DIR / "zoo_checks.csv", index=False)
    screen.to_csv(OUTPUT_DIR / "zoo_screen_report.csv", index=False)
    return checks, screen

In [ ]:
screen_start = time.perf_counter()

# Pixel L1 reference line of the screen (its blur penalty is close to 1 by construction).
frame = screen_network(lambda a, b: (a - b).abs().mean(), FILES["val"], device, crop=CROP_SIZE)
frame.to_csv(SCREEN_DIR / "pixel_l1.csv", index=False)
SCREEN_ROWS[("Pixel L1 (reference)", "-")] = {"Network": "Pixel L1 (reference)", "Matching": "-",
                                              **screen_summary(frame), "Parameters": 0, "trained": True,
                                              "screen_seconds": time.perf_counter() - screen_start}

for label, name in ZOO:
    try:
        evaluate_network(label, name)
    except Exception as error:      # a network that cannot be built must not hide the others
        FAILED[label] = repr(error)
        print(f"FAILED {label}: {error!r}")

record_seconds("screen_pretrained", time.perf_counter() - screen_start)
df_checks, df_screen = save_zoo_reports()
display(df_checks)
display(df_screen)
assert not FAILED, f"networks that could not be evaluated: {FAILED}"

# Week 2: Perceptual Losses and the CT Dose Network

## Step 2.2: Train the CT dose network
Six classes (five dose levels and normal dose), 128 x 128 patches from train-split patients
only, positions sampled **inside the body mask**. The old notebook drew positions uniformly
over the slice, so a large share of the patches were air.

In [ ]:
def body_mask(image: np.ndarray, threshold: float = BODY_THRESHOLD) -> np.ndarray:
    """Largest connected region above the threshold, with holes (lungs, bowel gas) filled."""
    labels, count = ndimage.label(image > threshold)
    if count == 0:
        return np.zeros(image.shape, dtype=bool)
    sizes = np.bincount(labels.ravel())[1:]
    return ndimage.binary_fill_holes(labels == 1 + int(sizes.argmax()))


def body_fraction_map(mask: np.ndarray, patch: int) -> np.ndarray:
    """Share of body pixels in the patch at every top-left position (integral image)."""
    integral = np.pad(mask.astype(np.float64), ((1, 0), (1, 0))).cumsum(0).cumsum(1)
    inside = integral[patch:, patch:] - integral[:-patch, patch:] - integral[patch:, :-patch] + integral[:-patch, :-patch]
    return inside / patch ** 2


def sample_body_patches(mask: np.ndarray, patch: int, count: int, rng: np.random.Generator,
                        min_fraction: float = MIN_BODY_FRACTION) -> list[tuple[int, int, float]]:
    """Top-left corners of `count` patches that lie inside the body mask.

    A position is valid when at least `min_fraction` of the patch is body. On a slice too small for
    that (neck, apex), the positions closest to the slice's best achievable fraction are used.
    """
    fraction = body_fraction_map(mask, patch)
    valid = fraction >= min_fraction
    if not valid.any():
        valid = fraction >= 0.95 * fraction.max()
    rows, cols = np.nonzero(valid)
    picks = rng.choice(len(rows), size=count, replace=len(rows) < count)
    return [(int(rows[i]), int(cols[i]), float(fraction[rows[i], cols[i]])) for i in picks]


DOSE_CLASS_KEYS = (*DOSE_KEYS, "gt_clipped")
DOSE_CLASS_NAMES = ["10k", "50k", "100k", "500k", "1000k", "clean"]


class DosePatches(Dataset):
    """One patch position per sample, shared by the six versions of the slice (one sample per class)."""

    def __init__(self, files: list[Path], seed: int, patch: int = DOSE_PATCH_SIZE,
                 per_slice: int = DOSE_PATCHES_PER_SLICE) -> None:
        rng = np.random.default_rng(seed)
        self.patch, self.samples, self.fractions = patch, [], []
        for path in tqdm(files, desc="body masks", leave=False):
            with np.load(path) as sample:
                mask = body_mask(sample["gt_clipped"])
            for top, left, fraction in sample_body_patches(mask, patch, per_slice, rng):
                self.fractions.append(fraction)
                self.samples.extend((path, top, left, label) for label in range(len(DOSE_CLASS_KEYS)))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        path, top, left, label = self.samples[index]
        with np.load(path) as sample:
            patch = sample[DOSE_CLASS_KEYS[label]][top:top + self.patch, left:left + self.patch].astype(np.float32)
        return torch.from_numpy(patch).unsqueeze(0), label


train_patches = DosePatches(FILES["train"], seed=SEED)
val_patches = DosePatches(FILES["val"], seed=SEED + 1)
dose_patients = {split: {patient_id(p) for p in FILES[split]} for split in ("train", "val")}
assert not dose_patients["train"] & dose_patients["val"], "patient leakage between dose-network splits"
print(f"train patches: {len(train_patches)} | val patches: {len(val_patches)} (6 classes per position)")
print(f"mean body fraction of a patch: train {np.mean(train_patches.fractions):.3f}, "
      f"val {np.mean(val_patches.fractions):.3f}; "
      f"below {MIN_BODY_FRACTION}: {np.mean(np.array(train_patches.fractions) < MIN_BODY_FRACTION):.1%} of train positions")

In [ ]:
# Look at the mask before trusting it: body outline in orange, sampled patches in blue.
fig, axes = plt.subplots(1, 4, figsize=(16, 4.3))
for axis, path in zip(axes, FILES["train"][:: max(1, len(FILES["train"]) // 4)]):
    with np.load(path) as sample:
        image = sample["gt_clipped"]
    mask = body_mask(image)
    axis.imshow(image, cmap="gray", vmin=0, vmax=1)
    axis.contour(mask, levels=[0.5], colors="tab:orange", linewidths=1)
    for top, left, _ in sample_body_patches(mask, DOSE_PATCH_SIZE, DOSE_PATCHES_PER_SLICE, np.random.default_rng(0)):
        axis.add_patch(plt.Rectangle((left, top), DOSE_PATCH_SIZE, DOSE_PATCH_SIZE, fill=False, color="tab:blue", lw=1))
    axis.set_title(f"body {mask.mean():.0%} of slice")
    axis.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
dose_report_path = DOSE_NET_DIR / "report.json"
DOSE_NET_DIR.mkdir(parents=True, exist_ok=True)

if DOSE_NET_PATH.exists() and dose_report_path.exists() and not FORCE_RERUN:
    dose_report = json.loads(dose_report_path.read_text())
    confusion = pd.read_csv(DOSE_NET_DIR / "confusion_matrix.csv", index_col=0).to_numpy()
    print("CT dose network: finished run found, loading it")
else:
    seed_everything(SEED)
    dose_net = CTDoseNet().to(device)
    patch_loader = make_loader(train_patches, 64, shuffle=True, seed=SEED)
    optimizer = torch.optim.Adam(dose_net.parameters(), lr=1e-3)
    dose_log, dose_start = [], time.perf_counter()
    for epoch in range(DOSE_NET_EPOCHS):
        dose_net.train()
        running, correct, seen = 0.0, 0, 0
        for patches, labels in patch_loader:
            patches, labels = patches.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = dose_net(patches)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()
            running += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            seen += labels.size(0)
        dose_log.append({"epoch": epoch + 1, "train_loss": running / seen, "train_accuracy": correct / seen})
        print(f"epoch {epoch + 1:02d}/{DOSE_NET_EPOCHS} | loss {running / seen:.4f} | accuracy {correct / seen:.2%}")
    dose_seconds = time.perf_counter() - dose_start
    torch.save(dose_net.state_dict(), DOSE_NET_PATH)
    pd.DataFrame(dose_log).to_csv(DOSE_NET_DIR / "train_log.csv", index=False)

    dose_net.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for patches, labels in make_loader(val_patches, 64):
            all_logits.append(dose_net(patches.to(device)).float().cpu())
            all_labels.append(labels)
    logits, labels = torch.cat(all_logits), torch.cat(all_labels)
    predictions = logits.argmax(1)
    classes = len(DOSE_CLASS_NAMES)
    confusion = np.bincount((labels * classes + predictions).numpy(), minlength=classes ** 2).reshape(classes, classes)

    # Clean vs 1000k as a two-way decision: only those patches, only those two logits.
    hardest = [DOSE_CLASS_NAMES.index("1000k"), DOSE_CLASS_NAMES.index("clean")]
    keep = torch.isin(labels, torch.tensor(hardest))
    two_way = torch.tensor(hardest)[logits[keep][:, hardest].argmax(1)]
    dose_report = {
        "val_accuracy": float((predictions == labels).float().mean()),
        "val_accuracy_clean_vs_1000k": float((two_way == labels[keep]).float().mean()),
        "train_seconds": dose_seconds, "gpu_hours": dose_seconds / 3600,
        "parameters": count_params(dose_net), "epochs": DOSE_NET_EPOCHS,
        "train_patches": len(train_patches), "val_patches": len(val_patches),
        "mean_body_fraction_train": float(np.mean(train_patches.fractions)),
        "mean_body_fraction_val": float(np.mean(val_patches.fractions)),
        "body_threshold": BODY_THRESHOLD, "min_body_fraction": MIN_BODY_FRACTION,
    }
    dose_report_path.write_text(json.dumps(dose_report, indent=2))
    pd.DataFrame(confusion, index=DOSE_CLASS_NAMES, columns=DOSE_CLASS_NAMES).to_csv(DOSE_NET_DIR / "confusion_matrix.csv")
    record_seconds("ct_dose_net_training", dose_seconds)
    free(dose_net)

support = confusion.sum(1)
precision = np.diag(confusion) / np.maximum(confusion.sum(0), 1)
recall = np.diag(confusion) / np.maximum(support, 1)
class_report = pd.DataFrame({"class": DOSE_CLASS_NAMES, "precision": precision, "recall": recall,
                             "f1": 2 * precision * recall / np.maximum(precision + recall, 1e-12),
                             "support": support})
class_report.to_csv(DOSE_NET_DIR / "classification_report.csv", index=False)
print(f"validation accuracy, 6 classes: {dose_report['val_accuracy']:.2%} (chance 16.7%)")
print(f"validation accuracy, clean vs 1000k: {dose_report['val_accuracy_clean_vs_1000k']:.2%} (chance 50%)")
print(f"training time: {dose_report['train_seconds']:.0f} s = {dose_report['gpu_hours']:.3f} GPU h")
display(class_report)

fig, axis = plt.subplots(figsize=(6, 5))
axis.imshow(confusion, cmap="Blues")
for i in range(len(DOSE_CLASS_NAMES)):
    for j in range(len(DOSE_CLASS_NAMES)):
        axis.text(j, i, int(confusion[i, j]), ha="center", va="center",
                  color="white" if confusion[i, j] > confusion.max() / 2 else "black")
axis.set_xticks(range(len(DOSE_CLASS_NAMES)), DOSE_CLASS_NAMES)
axis.set_yticks(range(len(DOSE_CLASS_NAMES)), DOSE_CLASS_NAMES)
axis.set_xlabel("predicted class")
axis.set_ylabel("true class")
axis.set_title("CT dose network, validation patches")
plt.tight_layout()
plt.show()

### Screen the frozen CT dose network (Step 1.4 rows for C9 and C10)
A blur penalty below 1 is a finding, not a bug; it is reported as measured.

In [ ]:
screen_start = time.perf_counter()
evaluate_network("CT dose network", "ct_dose_net")
record_seconds("screen_ct_dose_net", time.perf_counter() - screen_start)
df_checks, df_screen = save_zoo_reports()
display(df_screen[df_screen["Network"] == "CT dose network"])

## Step 2.1: Calibrate each loss weight by perceptual influence

$$\lambda_\phi = \frac{\Psi^\star}{1 - \Psi^\star} \cdot \frac{S_1}{S_\phi}$$

$S_1$ and $S_\phi$ are measured with the untrained seed-42 U-Net on the training split. Every
network sees the same slices in the same order and the same sequence of random 224 x 224 crops
(the crop rule of the training loss), so $S_1$ is identical in every row. C0 has no perceptual
term and keeps $\lambda = 0$. Only the configurations in `TRAIN_CONFIGS` are calibrated.

In [ ]:
@torch.no_grad()
def initial_loss_terms(model: nn.Module, distance: nn.Module, seed: int = SEED) -> tuple[float, float]:
    """Mean L1 term and mean feature distance over the training split at initialization."""
    crop_rng = torch.Generator().manual_seed(seed)             # same crops for every network
    pixel_sum, feature_sum, seen = 0.0, 0.0, 0
    for noisy, clean, _ in tqdm(make_loader(DATASETS["train"], BATCH_SIZE), desc="calibrate", leave=False):
        prediction, clean = model(noisy.to(device)).float(), clean.to(device)
        pixel_sum += F.l1_loss(prediction, clean).item() * noisy.size(0)
        feature_sum += distance(*random_crop_pair(prediction, clean, CROP_SIZE, crop_rng)).item() * noisy.size(0)
        seen += noisy.size(0)
    return pixel_sum / seen, feature_sum / seen


calibration_start = time.perf_counter()
seed_everything(SEED)
initial_model = UNet().to(device).eval()        # theta_0: the initialization every seed-42 run starts from

calibration_rows = [{"Configuration": "C0", "Label": CONFIG_SPECS["C0"][2], "Matching": "-",
                     "S_1": float("nan"), "S_phi": float("nan"), "lambda": 0.0, "influence": 0.0}]
for config_id in [c for c in CONFIG_SPECS if c != "C0" and c in TRAIN_CONFIGS]:
    name, matching, label = CONFIG_SPECS[config_id]
    distance = make_distance(name, matching).to(device)
    s_1, s_phi = initial_loss_terms(initial_model, distance)
    weight = TARGET_INFLUENCE / (1 - TARGET_INFLUENCE) * s_1 / s_phi
    calibration_rows.append({"Configuration": config_id, "Label": label, "Matching": matching, "S_1": s_1,
                             "S_phi": s_phi, "lambda": weight, "influence": weight * s_phi / (s_1 + weight * s_phi)})
    print(f"{config_id:>3} {label:<24} S_1 {s_1:.6f} | S_phi {s_phi:.6f} | lambda {weight:.4f}")
    free(distance)
free(initial_model)
record_seconds("calibration", time.perf_counter() - calibration_start)

df_calibration = pd.DataFrame(calibration_rows)
calibrated = df_calibration[df_calibration["Configuration"] != "C0"]
assert np.allclose(calibrated["S_1"], calibrated["S_1"].iloc[0], rtol=1e-6), "S_1 differs between rows"
df_calibration.loc[df_calibration["Configuration"] == "C0", "S_1"] = calibrated["S_1"].iloc[0]
df_calibration.to_csv(OUTPUT_DIR / "calibration_weights.csv", index=False)
LOSS_WEIGHTS = dict(zip(df_calibration["Configuration"], df_calibration["lambda"]))
display(df_calibration)

## Step 2.3: Seed 42 runs for C1 to C10
Every run uses the Step 2.1 weight and the Protocol budget. `ms per step` and GPU hours are
recorded per run. If one configuration diverges in its first epoch at $\Psi^\star = 0.95$,
lower `TARGET_INFLUENCE` for all configurations and recalibrate, never for one row only.

In [ ]:
RUN_STATUS = {}
for config_id in [c for c in TRAIN_CONFIGS if c != "C0"]:
    print(f"\n>>> {config_id}: {CONFIG_SPECS[config_id][2]} (lambda {LOSS_WEIGHTS[config_id]:.4f})")
    try:
        RESULTS[config_id], SUMMARIES[config_id] = train_config(config_id, LOSS_WEIGHTS[config_id])
        RUN_STATUS[config_id] = "finished"
    except Exception as error:      # keep going, but report the failure exactly as it happened
        RUN_STATUS[config_id] = f"failed: {error!r}"
        print(f"<<< {config_id} FAILED: {error!r}")
        if device.type == "cuda":
            torch.cuda.empty_cache()

display(pd.Series(RUN_STATUS, name="status").to_frame())
assert all(status == "finished" for status in RUN_STATUS.values()), "at least one configuration failed"

### Table 1 rows (seed 42, test split, 100k)
The spread is over test slices, not seeds.

In [ ]:
ORDER = [c for c in CONFIG_SPECS if c in SUMMARIES]
table1_rows = [{"Configuration": "LDCT input (100k)", **{m: df_baseline[m].mean() for m in METRICS},
                **{f"{m}_std": df_baseline[m].std(ddof=1) for m in METRICS}}]
for config_id in ORDER:
    summary = SUMMARIES[config_id]
    table1_rows.append({"Configuration": f"{config_id} {summary['label']}",
                        **{m: summary[f"{m}_mean"] for m in METRICS},
                        **{f"{m}_std": summary[f"{m}_std"] for m in METRICS},
                        "abs_1_minus_r_h": float((1 - RESULTS[config_id]["r_h"]).abs().mean()),
                        "ms_per_step": summary["ms_per_step"], "gpu_hours": summary["gpu_hours"],
                        "lambda": summary["loss_weight"], "best_epoch": summary["best_epoch"],
                        "nonfinite_steps": summary["nonfinite_steps"]})
df_table1 = pd.DataFrame(table1_rows)
df_table1.to_csv(OUTPUT_DIR / f"table1_seed{SEED}.csv", index=False)
display(df_table1[["Configuration", *METRICS, "abs_1_minus_r_h", "ms_per_step", "gpu_hours", "nonfinite_steps"]])

### Visual check of one test slice
All panels share one display window (0 to 1), so brightness is comparable between panels.

In [ ]:
noisy_slice, clean_slice, stem = DATASETS["test"][0]
panels = [("100k input", noisy_slice[0].numpy()), ("normal dose", clean_slice[0].numpy())]
viewer = UNet().to(device).eval()
for config_id in ORDER:
    viewer.load_state_dict(torch.load(RUN_ROOT / f"train_{config_id}" / "best.pth", map_location=device))
    with torch.no_grad():
        panels.append((config_id, viewer(noisy_slice[None].to(device)).clamp(0, 1)[0, 0].float().cpu().numpy()))
free(viewer)
columns = 5
fig, axes = plt.subplots(int(np.ceil(len(panels) / columns)), columns, figsize=(4 * columns, 4 * int(np.ceil(len(panels) / columns))))
for axis in np.ravel(axes):
    axis.axis("off")
for axis, (title, image) in zip(np.ravel(axes), panels):
    axis.imshow(image, cmap="gray", vmin=0, vmax=1)
    axis.set_title(title)
plt.suptitle(stem)
plt.tight_layout()
plt.show()

## Table 2 values and the Compute Budget
Per trained network: parameters, $\lambda_\phi$, ms per step, $M_\phi$, $B_\phi$. The Compute
Budget lists what Weeks 1 and 2 measured and the total that the 3-seed protocol implies.

In [ ]:
ZOO_LABEL = {key: label for label, key in ZOO} | {"ct_dose_net": "CT dose network"}
screen_index = df_screen.set_index(["Network", "Matching"])
check_index = df_checks.set_index(["Network", "Matching"])
table2_rows = []
for config_id in [c for c in ORDER if c != "C0"]:
    name, matching, label = CONFIG_SPECS[config_id]
    key = (ZOO_LABEL[name], matching)
    table2_rows.append({"Configuration": config_id, "Network": ZOO_LABEL[name], "Matching": matching,
                        "Parameters": int(check_index.loc[key, "Parameters"]),
                        "lambda": SUMMARIES[config_id]["loss_weight"],
                        "ms_per_step": SUMMARIES[config_id]["ms_per_step"],
                        "ms_per_batch_of_8": check_index.loc[key, "ms_per_batch_of_8"],
                        "M_phi": screen_index.loc[key, "ordering_M"],
                        "B_phi": screen_index.loc[key, "blur_penalty_B"]})
df_table2 = pd.DataFrame(table2_rows)
df_table2.to_csv(OUTPUT_DIR / "table2_feature_networks.csv", index=False)
display(df_table2)

In [ ]:
timings = json.loads(TIMINGS_PATH.read_text()) if TIMINGS_PATH.exists() else {}
unet_hours = {c: SUMMARIES[c]["gpu_hours"] for c in ORDER}
other_hours = {name: seconds / 3600 for name, seconds in timings.items()}
other_hours.setdefault("ct_dose_net_training", dose_report["gpu_hours"])
seeds_in_protocol = 3
all_trained = set(ORDER) == set(CONFIG_SPECS)


def hours(config_id: str) -> str:
    return f"{unet_hours[config_id]:.3f}" if config_id in unet_hours else "-"


budget_rows = [
    ("Trained configurations", str(len(CONFIG_SPECS))),
    ("Seeds", str(seeds_in_protocol)),
    ("Full U-Net runs", str(len(CONFIG_SPECS) * seeds_in_protocol)),
    ("CT dose network pretraining, GPU hours (measured)", f"{other_hours['ct_dose_net_training']:.3f}"),
    ("Screen (Step 1.4), GPU hours (measured)",
     f"{other_hours.get('screen_pretrained', float('nan')) + other_hours.get('screen_ct_dose_net', float('nan')):.3f}"),
    ("Calibration (Step 2.1), GPU hours (measured)", f"{other_hours.get('calibration', float('nan')):.3f}"),
    ("GPU hours per U-Net run, C0 (measured in Step 1.2)", hours("C0")),
    ("GPU hours per U-Net run, C7 MedSigLIP (measured in Step 2.3)", hours("C7")),
    (f"GPU hours of the seed {SEED} U-Net runs ({len(ORDER)} of {len(CONFIG_SPECS)})", f"{sum(unet_hours.values()):.3f}"),
    ("GPU hours used in Weeks 1 and 2", f"{sum(unet_hours.values()) + sum(other_hours.values()):.3f}"),
    ("Total GPU hours of the protocol (3 seeds, from the seed 42 runs)",
     f"{seeds_in_protocol * sum(unet_hours.values()) + sum(other_hours.values()):.3f}" if all_trained else "- (not every configuration was run)"),
    ("GPU", f"{environment['gpu']} ({environment['gpu_memory_gb']} GB)"),
    ("Largest peak GPU memory of a run, GB", f"{max(SUMMARIES[c]['peak_gpu_memory_gb'] for c in ORDER):.1f}" if ORDER else "-"),
    ("GPU hours available in Weeks 1 to 3", "fill in by hand"),
]
df_budget = pd.DataFrame(budget_rows, columns=["Item", "Value"])
df_budget.to_csv(OUTPUT_DIR / "compute_budget.csv", index=False)
display(df_budget)

per_run = pd.DataFrame([{"Configuration": c, "ms_per_step": SUMMARIES[c]["ms_per_step"],
                         "train_seconds": SUMMARIES[c]["train_seconds"], "test_seconds": SUMMARIES[c]["test_seconds"],
                         "gpu_hours": SUMMARIES[c]["gpu_hours"], "peak_gpu_memory_gb": SUMMARIES[c]["peak_gpu_memory_gb"],
                         "batch_size": SUMMARIES[c]["batch_size"]} for c in ORDER])
per_run.to_csv(OUTPUT_DIR / "compute_per_run.csv", index=False)
display(per_run)

## Tables for `paper_plan.md` and the GitHub issue comment
One Markdown file with every Week 1 and Week 2 table of the plan, filled from the runs above.

In [ ]:
def mean_std(mean: float, std: float, digits: int) -> str:
    return f"{mean:.{digits}f} ± {std:.{digits}f}"


plan_table1 = pd.DataFrame([{
    "Configuration (seed 42, test, 100k)": row["Configuration"],
    "PSNR (dB)": mean_std(row["psnr"], row["psnr_std"], 2), "SSIM": mean_std(row["ssim"], row["ssim_std"], 4),
    "LPIPS": mean_std(row["lpips"], row["lpips_std"], 4), "DISTS": mean_std(row["dists"], row["dists_std"], 4),
    "R_H": mean_std(row["r_h"], row["r_h_std"], 4),
    "ms per step": "-" if pd.isna(row.get("ms_per_step")) else f"{row['ms_per_step']:.0f}",
} for row in table1_rows])
plan_screen = df_screen[["Network", "Matching", "ordering_M", "dose_ratio_10k_1000k", "blur_penalty_B", "Parameters"]]
plan_checks = df_checks[["Network", "Matching", "distance_at_equality", "gradient_reaches_prediction", "ms_per_batch_of_8"]]
plan_calibration = df_calibration[["Configuration", "Label", "S_1", "S_phi", "lambda"]]
plan_dose = pd.DataFrame([
    {"Check": "Val accuracy, 6 classes", "Observed": f"{dose_report['val_accuracy']:.2%}"},
    {"Check": "Val accuracy, clean vs 1000k", "Observed": f"{dose_report['val_accuracy_clean_vs_1000k']:.2%}"},
    {"Check": "Screen ordering M_phi (pointwise / statistics)",
     "Observed": " / ".join(f"{screen_index.loc[('CT dose network', m), 'ordering_M']:.4f}" for m in ("pointwise", "statistics"))},
    {"Check": "Screen blur penalty B_phi (pointwise / statistics)",
     "Observed": " / ".join(f"{screen_index.loc[('CT dose network', m), 'blur_penalty_B']:.4f}" for m in ("pointwise", "statistics"))},
    {"Check": "Mean body fraction of a training patch", "Observed": f"{dose_report['mean_body_fraction_train']:.3f}"},
])

sections = [
    ("Environment", pd.DataFrame(list(environment.items()), columns=["Item", "Value"])),
    ("Step 1.1 checks", step_1_1),
    ("Step 1.3 distance checks", plan_checks),
    ("Step 1.4 training-free screen (validation split)", plan_screen),
    ("Step 2.1 calibration", plan_calibration),
    ("Step 2.2 CT dose network checks", plan_dose),
    ("Step 2.2 CT dose network, per class", class_report),
    ("Step 1.2 and Step 2.3: Table 1 rows, seed 42 (spread over test slices)", plan_table1),
    ("Table 2 values per trained network", df_table2),
    ("Compute Budget", df_budget),
]
report = "\n\n".join(f"### {title}\n\n{to_markdown(frame)}" for title, frame in sections)
(OUTPUT_DIR / "week1_week2_tables.md").write_text(report + "\n")
display(Markdown(report))

In [ ]:
# Optional backup of every output to Google Drive.
if DRIVE_DIR is not None and DRIVE_DIR.parent.exists():
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    archive = shutil.make_archive(str(DRIVE_DIR / f"outputs_seed{SEED}"), "zip", str(OUTPUT_DIR))
    print("backup written to", archive)
else:
    print("no Drive folder configured; outputs stay in", OUTPUT_DIR)